Configure the local data snapshots as described in the README before running this notebook.

# BOVA11 volatility and risk research lab

Forecasts target the next exchange session. Configure the private snapshot as described in the README.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","arch","hmmlearn","xgboost","exchange-calendars"])

In [ ]:
from retratos_locais import ler_retrato_local

BASES = ler_retrato_local('BOVA11_Volatility_Research_Lab.ipynb', 'BASES')


## 1. Data and forecast horizon

The archived data span November 2020 through 5 November 2026. BOVA11 uses adjusted closing prices. Set `ATIVO` to `IBOV` to analyze the index.

In [ ]:
import base64, io, zlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy import stats
from scipy.special import logsumexp
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

ATIVO = "BOVA11"          # BOVA11 ou IBOV
INICIO_TESTE = "2025-01-01"
CSV_RV = globals().get("CSV_RV")             # Opcional: arquivo Date,RV com RV intradiária diária validada.
SAIDA = Path("resultados_risco")/ATIVO
SAIDA.mkdir(parents=True,exist_ok=True)
EPS = 1e-10
SEED = 42
PASSO_REAJUSTE = 63

def ler(nome):
    bruto = zlib.decompress(base64.b64decode(BASES[nome]))
    df = pd.read_csv(io.BytesIO(bruto),index_col="Date",parse_dates=True)
    df.index = df.index.astype("datetime64[ns]")
    assert df.index.is_unique and df.index.is_monotonic_increasing
    return df

assert ATIVO in {"BOVA11","IBOV"}
ibov = ler("ibov")
try:
    from calendario_mercado import calendario_e_sessoes
except ModuleNotFoundError as erro:
    if erro.name != "calendario_mercado":
        raise
    import exchange_calendars as xcals

    # Fallback gerado a partir de calendario_mercado.py.
    def calendario_e_sessoes(nome, inicio, fim):
        """Retorna o calendário e os rótulos de pregão entre duas datas inclusivas."""
        inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
        if any(pd.isna(dia) or dia.tz is not None or dia != dia.normalize()
               for dia in (inicio, fim)) or inicio > fim:
            raise ValueError("Informe datas locais, sem horário, em ordem crescente.")
        margem = pd.Timedelta(days=15)
        calendario = xcals.get_calendar(nome, start=inicio - margem, end=fim + margem)
        sessoes = calendario.sessions_in_range(inicio, fim)
        if sessoes.tz is not None:
            sessoes = sessoes.tz_localize(None)
        return calendario, sessoes

cal, sessoes = calendario_e_sessoes("BVMF", ibov.index.min(), ibov.index.max())
bruto = ler({"BOVA11":"bova11","IBOV":"ibov"}[ATIVO])
assert bruto.index.isin(sessoes).all()
coluna = "Adj Close" if ATIVO=="BOVA11" else "Close"
preco = bruto[coluna].reindex(sessoes)
assert np.isfinite(preco.dropna()).all() and preco.dropna().gt(0).all()
r = np.log(preco).diff()
variancia_obs = r.pow(2)
origem = pd.Series(sessoes,index=sessoes)
data_alvo = origem.shift(-1)
primeiro_teste = pd.Timestamp(INICIO_TESTE)
teste = sessoes[(data_alvo>=primeiro_teste).to_numpy()]
origens_previsao = teste.append(pd.DatetimeIndex([sessoes[-1]]))
assert len(teste)>=40
def carregar_rv_validada(caminho, ativo, sessoes, primeiro_teste, teste):
    import json

    manifesto = Path(caminho).with_name("manifesto_rv.json")
    if manifesto.exists():
        procedencia = json.loads(manifesto.read_text(encoding="utf-8"))
        if procedencia["ativo"] != ativo:
            raise ValueError("A variância realizada é de outro ativo.")
    tabela = pd.read_csv(caminho, index_col="Date", parse_dates=True)
    if tabela.empty:
        raise ValueError("O CSV de RV não contém pregões aprovados.")
    if (not isinstance(tabela.index,pd.DatetimeIndex) or not tabela.index.is_unique
        or not tabela.index.isin(sessoes).all()):
        raise ValueError("A RV exige datas únicas dentro do calendário do retrato.")
    rv = pd.to_numeric(tabela.RV, errors="raise").reindex(sessoes)
    if not np.isfinite(rv.dropna()).all() or rv.dropna().lt(0).any():
        raise ValueError("A RV precisa ser finita e não negativa.")
    janelas = rv.rolling(22).count().eq(22)
    if janelas.loc[janelas.index < primeiro_teste].sum() < 100:
        raise ValueError("RV precisa de histórico suficiente antes do teste: ao menos 100 janelas válidas de 22 sessões.")
    if not (janelas & rv.shift(-1).notna()).reindex(teste,fill_value=False).any():
        raise ValueError("A RV precisa de pelo menos uma origem de teste com 22 sessões e alvo observado em D+1.")
    return rv


rv = carregar_rv_validada(CSV_RV, ATIVO, sessoes, primeiro_teste, teste) if CSV_RV is not None else None

print(f"{ATIVO} | {sessoes[0].date()} a {sessoes[-1].date()} | {preco.isna().sum()} sessões sem preço")
print("Datas sem preço:", [str(d.date()) for d in preco.index[preco.isna()]])
print("Retornos calculados:",r.notna().sum(),"| primeiras origens de teste:",teste[0].date())

## 2. Historical volatility, drawdown, and tails

Drawdown is the ratio of the observed price to its running peak, minus one. Annualized rolling volatility uses 252 sessions. Autocorrelations are descriptive.

In [ ]:
capital = preco/preco.dropna().iloc[0]
drawdown = capital/capital.cummax()-1
pior = drawdown.idxmin()
pico = capital.loc[:pior].idxmax()
recuperacoes = capital.loc[pior:].dropna()
recuperacoes = recuperacoes.loc[recuperacoes>=capital.loc[pico]]
recuperacao = recuperacoes.index[0] if len(recuperacoes) else pd.NaT
r_limpo = r.dropna()
estat = pd.Series({"volatilidade anualizada":r_limpo.std()*np.sqrt(252),
                  "máximo drawdown":drawdown.min(),"data do pico":pico.date(),
                  "data do fundo":pior.date(),"recuperação":recuperacao,
                  "assimetria dos retornos":r_limpo.skew(),"curtose em excesso":r_limpo.kurt()})
display(estat)
fig,axs = plt.subplots(2,2,figsize=(13,7))
capital.plot(ax=axs[0,0],title="Capital por unidade inicial")
drawdown.plot(ax=axs[0,1],title="Drawdown desde o último pico",color="firebrick")
axs[0,1].yaxis.set_major_formatter(PercentFormatter(1))
r.plot(ax=axs[1,0],title="Retornos logarítmicos diários",lw=.65)
axs[1,0].yaxis.set_major_formatter(PercentFormatter(1))
r.rolling(21).std().mul(np.sqrt(252)).plot(ax=axs[1,1],title="Volatilidade móvel anualizada · 21 pregões")
axs[1,1].yaxis.set_major_formatter(PercentFormatter(1))
fig.tight_layout()
fig.savefig(SAIDA/"painel_historico.png",dpi=140)
plt.show()
# ACF em um bloco contínuo evita tratar pregões separados por lacunas como vizinhos.
blocos = [s.dropna() for _,s in r.groupby(r.isna().cumsum())]
contiguo = max(blocos,key=len)
from statsmodels.graphics.tsaplots import plot_acf
fig,ax = plt.subplots(1,2,figsize=(12,3.5))
plot_acf(contiguo,lags=25,ax=ax[0],title="ACF dos retornos")
plot_acf(contiguo.pow(2),lags=25,ax=ax[1],title="ACF dos retornos ao quadrado")
plt.tight_layout()
plt.show()
display(acorr_ljungbox(contiguo.pow(2),lags=[5,10,20],return_df=True))
drawdown.rename("drawdown").to_csv(SAIDA/"drawdown.csv",index_label="data")

## 3. Baselines, GARCH, EGARCH, GJR, and HMM

The historical baseline averages 21 squared returns. EWMA uses 0.94 decay. GARCH-family fits use zero mean and Student-t innovations.

In [ ]:
previsoes = pd.DataFrame(index=sessoes)
previsoes["Histórica 21"] = variancia_obs.rolling(21).mean()
# EWMA reinicia após retorno ausente, sem preencher a lacuna.
previsoes["EWMA 0,94"] = variancia_obs.groupby(r.isna().cumsum()).transform(
    lambda s:s.ewm(alpha=.06,adjust=False,min_periods=21).mean()).where(r.notna())

historico_pre = r.loc[r.index<primeiro_teste]
amostra = max([s.dropna() for _,s in historico_pre.groupby(historico_pre.isna().cumsum())],key=len)
assert len(amostra)>=252, "O ajuste inicial precisa de 252 retornos consecutivos."
amostra_pct = amostra*100
config_garch = {"GARCH(1,1)":("GARCH",0),"EGARCH":("EGARCH",1),"GJR-GARCH":("GARCH",1)}
params = {}
diagnosticos = []
for nome,(familia,assimetria) in config_garch.items():
    ajuste = arch_model(amostra_pct,mean="Zero",vol=familia,p=1,o=assimetria,q=1,dist="t").fit(
        disp="off",show_warning=False)
    if ajuste.convergence_flag:
        raise RuntimeError(f"{nome} não convergiu no ajuste inicial.")
    params[nome] = ajuste.params
    diagnosticos.append({"modelo":nome,"n_ajuste":len(amostra),"fim_ajuste":amostra.index.max().date(),
                         "nu":ajuste.params["nu"]})

# Uma previsão no fechamento t utiliza retornos observados até t.
# O bloco mais recente reinicia o filtro após lacunas no histórico.
r_pct = r*100
for nome,(familia,assimetria) in config_garch.items():
    saida = {}
    for _,bloco in r_pct.groupby(r.isna().cumsum()):
        bloco = bloco.dropna()
        for dia in bloco.index.intersection(origens_previsao):
            hist = bloco.loc[:dia]
            if len(hist)<22:
                continue
            modelo = arch_model(hist.to_numpy(),mean="Zero",vol=familia,p=1,o=assimetria,q=1,dist="t")
            fit = modelo.fix(params[nome])
            saida[dia] = fit.forecast(horizon=1,reindex=False).variance.iloc[-1,0]/1e4
    previsoes[nome] = pd.Series(saida,dtype=float)
display(pd.DataFrame(diagnosticos))

hmm = GaussianHMM(n_components=2,covariance_type="diag",n_iter=250,random_state=SEED).fit(
    amostra_pct.to_numpy()[:,None])
if not hmm.monitor_.converged:
    warnings.warn("HMM sem indicação de convergência; interprete como experimental.")
media_estado = hmm.means_.ravel()
var_estado = np.maximum(hmm.covars_.reshape(2,-1)[:,0],1e-8)
prob = hmm.startprob_
segundo_momento,regime = [],[]
for obs in r_pct:
    if pd.notna(obs):
        logp = np.log(np.maximum(prob,1e-300))-.5*(np.log(var_estado)+(obs-media_estado)**2/var_estado)
        prob = np.exp(logp-logsumexp(logp))
    prob = prob @ hmm.transmat_
    segundo_momento.append(prob @ (var_estado+media_estado**2)/1e4)
    regime.append(prob[np.argmax(var_estado)])
previsoes["HMM 2 estados"] = segundo_momento
prob_regime = pd.Series(regime,index=sessoes).where(sessoes>=primeiro_teste-pd.Timedelta(days=7))

## 4. Walk-forward HAR and XGBoost

HAR combines variance proxies at 1, 5, and 22 sessions. XGBoost adds recent returns and external variables dated strictly before the forecast origin. Both are refitted every 63 origins.

In [ ]:
def fatores_har(serie):
    return pd.DataFrame({f"m{w}":serie.rolling(w).mean() for w in [1,5,22]})

def arvores():
    return XGBRegressor(objective="reg:gamma",n_estimators=160,max_depth=2,min_child_weight=12,
                        learning_rate=.035,subsample=.85,colsample_bytree=.85,
                        random_state=SEED,n_jobs=2)

def walk_forward(serie,extras,origens):
    h = fatores_har(serie)
    alvo = serie.shift(-1)
    x = h.join(extras)
    har,boost = {},{}
    for inicio in range(0,len(origens),PASSO_REAJUSTE):
        bloco = origens[inicio:inicio+PASSO_REAJUSTE]
        # O último alvo de treino precisa estar disponível no primeiro fechamento do bloco.
        treino_ok = x.notna().all(axis=1)&alvo.notna()&(data_alvo<=bloco[0])
        idx_treino = x.index[treino_ok]
        idx_pred = bloco.intersection(x.index[x.notna().all(axis=1)])
        if len(idx_treino)<100 or not len(idx_pred):
            continue
        assert data_alvo.loc[idx_treino].max()<=idx_pred.min()
        yy = alvo.loc[idx_treino].clip(lower=EPS)
        # HAR em níveis, com piso positivo. Treino e previsão usam a mesma escala.
        lin = LinearRegression().fit(h.loc[idx_treino]*1e4,yy*1e4)
        xgb = arvores().fit(x.loc[idx_treino],yy*1e4)
        har.update(zip(idx_pred,np.maximum(lin.predict(h.loc[idx_pred]*1e4)/1e4,EPS)))
        boost.update(zip(idx_pred,np.maximum(xgb.predict(x.loc[idx_pred])/1e4,EPS)))
    return pd.DataFrame({"HAR":pd.Series(har,dtype=float),"XGBoost":pd.Series(boost,dtype=float)})

extras = pd.DataFrame({"retorno":r,"retorno_negativo":r.clip(upper=0),
                       "abs_ret_5":r.abs().rolling(5).mean()},index=sessoes)
for nome in ["ibov","sp500","dolar","vix","petroleo","ewz"]:
    if ATIVO=="IBOV" and nome=="ibov":
        continue
    externo = ler(nome).Close.dropna()
    assert externo.gt(0).all()
    features = pd.DataFrame({f"{nome}_ret1":np.log(externo).diff(),
                             f"{nome}_vol5":np.log(externo).diff().rolling(5).std(),
                             "fonte":externo.index})
    alinha = pd.merge_asof(pd.DataFrame(index=sessoes),features,left_index=True,right_index=True,
                           allow_exact_matches=False,tolerance=pd.Timedelta(days=7))
    assert (alinha.fonte.dropna()<alinha.fonte.dropna().index).all()
    extras = extras.join(alinha.drop(columns="fonte"))
ml = walk_forward(variancia_obs,extras,origens_previsao)
previsoes["HAR · proxy diária"] = ml.HAR
previsoes["XGBoost"] = ml.XGBoost

## 5. Model comparison

The next-session target is the squared log return, a noisy variance proxy. Models are compared on common dates. QLIKE is log predicted variance plus observed variance divided by predicted variance.

In [ ]:
def comparar(forecasts,observada,origens):
    quadro = forecasts.reindex(origens).join(observada.shift(-1).rename("observada"))
    disponiveis = quadro.notna().sum()
    comum = quadro.dropna()
    if comum.empty:
        raise ValueError("Nenhuma data em comum para os modelos.")
    vv = comum.observada.to_numpy()
    linhas = []
    for nome in forecasts:
        h = comum[nome].clip(lower=EPS).to_numpy()
        linhas.append({"modelo":nome,"QLIKE":np.mean(np.log(h)+vv/h),
                       "MAE_var":np.mean(np.abs(h-vv)),"RMSE_var":np.sqrt(np.mean((h-vv)**2)),
                       "observada_prevista":np.sum(vv)/np.sum(h),
                       "n_disponivel":int(disponiveis[nome]),"n_comum":len(comum)})
    return pd.DataFrame(linhas).sort_values("QLIKE"),comum

ranking,comum = comparar(previsoes,variancia_obs,teste)
display(ranking.style.format({"QLIKE":"{:.3f}","MAE_var":"{:.3e}","RMSE_var":"{:.3e}",
                              "observada_prevista":"{:.2f}"}))
ranking.to_csv(SAIDA/"ranking_volatilidade.csv",index=False)
previsoes_exportar = previsoes.loc[teste].join(data_alvo.rename("data_alvo")).join(
    variancia_obs.shift(-1).rename("proxy_var_observada"))
previsoes_exportar.to_csv(SAIDA/"previsoes_D1.csv",index_label="origem")
datas_plot = pd.DatetimeIndex(data_alvo.loc[teste])
comum_plot = comum.reindex(teste)
fig,ax = plt.subplots(3,1,figsize=(13,9),sharex=True)
ax[0].plot(datas_plot,np.sqrt(comum_plot.observada*252),label="Proxy observada de D+1",color="0.65",lw=.65)
for nome in ["Histórica 21","GARCH(1,1)","HAR · proxy diária","XGBoost"]:
    ax[0].plot(datas_plot,np.sqrt(comum_plot[nome].clip(lower=EPS)*252),label=nome,lw=1)
ax[0].set_title("Volatilidade em escala anual · por data do alvo")
ax[0].yaxis.set_major_formatter(PercentFormatter(1))
ax[0].legend(ncol=3)
ax[1].plot(datas_plot,prob_regime.reindex(teste),color="#805ad5")
ax[1].set_title("HMM · probabilidade filtrada de maior volatilidade em D+1")
ax[1].set_ylim(0,1)
ax[1].yaxis.set_major_formatter(PercentFormatter(1))
for nome in ["EWMA 0,94","GARCH(1,1)","XGBoost"]:
    h = comum[nome].clip(lower=EPS)
    ax[2].plot(datas_plot,(np.log(h)+comum.observada/h).rolling(42).mean().reindex(teste),label=nome)
ax[2].set_title("QLIKE médio em 42 previsões · menor é melhor")
ax[2].legend()
fig.tight_layout()
fig.savefig(SAIDA/"painel_previsoes.png",dpi=140)
plt.show()

## 6. VaR and Expected Shortfall

Loss is the negative log return. VaR is a loss quantile. Expected Shortfall averages the tail, with fractional mass at the empirical quantile boundary.

In [ ]:
# Perdas em unidades de log-retorno. VaR é quantil e CVaR é média da cauda superior.
perda = -r
perda_d1 = perda.shift(-1)
def es_empirico(v,alpha):
    v = np.sort(v[np.isfinite(v)])[::-1]
    massa = len(v)*alpha
    inteiros = int(np.floor(massa))
    return (v[:inteiros].sum()+(massa-inteiros)*v[inteiros])/massa

risco = pd.DataFrame(index=sessoes)
resultados_cauda = []
nu = float(params["GARCH(1,1)"]["nu"])
sigma = np.sqrt(previsoes["GARCH(1,1)"].clip(lower=EPS))
assert nu>2
for nivel in [.95,.99]:
    alpha = 1-nivel
    rotulo = int(round(nivel*100))
    janela = perda.rolling(252,min_periods=200)
    risco[f"VaR_hist_{rotulo}"] = janela.quantile(nivel)
    risco[f"ES_hist_{rotulo}"] = janela.apply(lambda v:es_empirico(v,alpha),raw=True)
    q = stats.t.ppf(nivel,df=nu)
    escala = np.sqrt((nu-2)/nu)
    risco[f"VaR_t_{rotulo}"] = sigma*escala*q
    risco[f"ES_t_{rotulo}"] = sigma*escala*(nu+q*q)/(nu-1)*stats.t.pdf(q,df=nu)/alpha
    cols = [f"{met}_{mod}_{rotulo}" for mod in ["hist","t"] for met in ["VaR","ES"]]
    validos = risco.loc[teste,cols].join(perda_d1.rename("perda")).dropna()
    for mod in ["hist","t"]:
        v,es = validos[f"VaR_{mod}_{rotulo}"],validos[f"ES_{mod}_{rotulo}"]
        viola = validos.perda>v
        # Pinball menor é melhor para o quantil da perda.
        erro = validos.perda-v
        resultados_cauda.append({"modelo":mod,"nivel":nivel,"n_comum":len(validos),
            "violacoes":int(viola.sum()),"taxa_violacao":viola.mean(),"taxa_esperada":alpha,
            "pinball_VaR":np.maximum(nivel*erro,(nivel-1)*erro).mean(),
            "perda_media_quando_viola":validos.perda.loc[viola].mean(),
            "ES_medio_quando_viola":es.loc[viola].mean()})
tabela_risco = pd.DataFrame(resultados_cauda)
display(tabela_risco)
tabela_risco.to_csv(SAIDA/"backtest_VaR_ES.csv",index=False)
risco.loc[teste].join(data_alvo.rename("data_alvo")).join(perda_d1.rename("perda_observada")).to_csv(
    SAIDA/"VaR_ES_D1.csv",index_label="origem")
ult = risco.loc[[sessoes[-1]]].dropna()
if not ult.empty:
    data_proxima = cal.next_session(sessoes[-1])
    print(f"Estimativa a partir do último fechamento do retrato: {sessoes[-1].date()} → {data_proxima.date()}")
    ultima_vol = np.sqrt(previsoes.loc[sessoes[-1]].clip(lower=EPS)*252)
    display(ultima_vol.rename("volatilidade anualizada prevista").to_frame().style.format("{:.2%}"))
if not ult.empty:
    display(ult.T.rename(columns={ult.index[0]:"estimativa diária"}).style.format("{:.2%}"))
fig,ax = plt.subplots(figsize=(13,4))
ax.plot(data_alvo.loc[teste],perda_d1.loc[teste],label="Perda observada",lw=.7,color=".5")
for nome in ["VaR_hist_95","VaR_t_95","ES_t_95"]:
    ax.plot(data_alvo.loc[teste],risco.loc[teste,nome],label=nome,lw=1)
ax.set_title("Cauda de perdas em D+1 · VaR e CVaR/ES de 95%")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend()
fig.tight_layout()
fig.savefig(SAIDA/"painel_caudas.png",dpi=140)
plt.show()
# Faixas centrais de retorno implicadas pela Student-t simétrica de média zero.
intervalos = risco.loc[teste,["VaR_t_95","VaR_t_99"]].join(r.shift(-1).rename("retorno")).dropna()
faixas = []
for cauda,central in [(95,.90),(99,.98)]:
    limites = intervalos[f"VaR_t_{cauda}"]
    faixas.append({"cobertura_nominal":central,"n":len(limites),
        "cobertura_observada":intervalos.retorno.abs().le(limites).mean()})
tabela_faixas = pd.DataFrame(faixas)
display(tabela_faixas)
tabela_faixas.to_csv(SAIDA/"cobertura_intervalos_GARCH.csv",index=False)
fig,ax = plt.subplots(figsize=(13,3.6))
datas = data_alvo.loc[teste]
intervalos_plot = intervalos.reindex(teste)
for cauda,alpha_plot in [(99,.16),(95,.25)]:
    lim = intervalos_plot[f"VaR_t_{cauda}"]
    ax.fill_between(datas,-lim,lim,alpha=alpha_plot,label=f"Faixa central de {2*cauda-100}%")
ax.plot(datas,intervalos_plot.retorno,color=".3",lw=.65,label="Retorno observado")
ax.set_title("GARCH Student-t · faixas centrais de retorno em D+1")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(SAIDA/"painel_intervalos.png",dpi=140)
plt.show()

## 7. Optional HAR-RV with intraday observations

Set `CSV_RV` to a CSV containing `Date,RV` for the selected asset. RV is the session sum of squared intraday log returns. The notebook checks coverage before fitting.

In [ ]:
if CSV_RV is None:
    print("Seção opcional. Informe CSV_RV para avaliar HAR-RV com variância realizada intradiária.")
else:
    pred_rv = walk_forward(rv,pd.DataFrame(index=sessoes),teste)
    pred_rv = pred_rv.rename(columns={"HAR":"HAR-RV","XGBoost":"XGBoost RV"})
    pred_rv["Média RV 22"] = rv.rolling(22).mean()
    ranking_rv,comum_rv = comparar(pred_rv,rv,teste)
    display(ranking_rv)
    ranking_rv.to_csv(SAIDA/"ranking_RV_intradiaria.csv",index=False)
    painel_rv = comum_rv.rename(columns={"observada":"RV_observada_D1"}).join(data_alvo.rename("data_alvo"))
    painel_rv.to_csv(SAIDA/"previsoes_RV_intradiaria.csv",index_label="origem")

## 8. P1.8 · Matched-date VaR and Expected Shortfall validation

The historical and Student-t GARCH methods are compared only where both have forecasts and an observed loss. Exact binomial tests assess coverage, and paired scores compare the joint tail forecasts.

In [ ]:
# Fallback gerado a partir de risco_estatistico.py para uso independente no Colab.
try:
    from risco_estatistico import adjacencia_sessoes, intervalo_media_pareada, perda_fz0, agrupamento_violacoes
except ModuleNotFoundError as erro:
    if erro.name!="risco_estatistico":
        raise
    def adjacencia_sessoes(sessoes, datas):
        """Marca vizinhos de calendário sem unir os dois lados de uma lacuna."""
        posicoes = sessoes.get_indexer(datas)
        passos = np.diff(posicoes)
        if len(posicoes)==0 or (posicoes<0).any() or (passos<=0).any():
            raise ValueError("A amostra precisa de sessões conhecidas, únicas e em ordem.")
        return passos==1

    def intervalo_media_pareada(diferenca,adjacentes,bloco=10,rep=3000,semente=42):
        """Bootstrap circular dentro de cada trecho contínuo, com tamanho preservado."""
        diferenca = np.asarray(diferenca,dtype=float)
        adjacentes = np.asarray(adjacentes,dtype=bool)
        if (diferenca.ndim!=1 or not len(diferenca) or not np.isfinite(diferenca).all()
            or adjacentes.shape!=(len(diferenca)-1,) or bloco<1 or rep<1):
            raise ValueError("Amostra pareada, adjacência ou configuração inválida.")
        rng = np.random.default_rng(semente)
        cortes = np.flatnonzero(~adjacentes)+1
        somas = np.zeros(rep)
        for seg in np.split(diferenca,cortes):
            k = min(bloco,len(seg))
            inicios = rng.integers(len(seg),size=(rep,int(np.ceil(len(seg)/k))))
            indices = ((inicios[:,:,None]+np.arange(k))%len(seg)).reshape(rep,-1)
            somas += seg[indices[:,:len(seg)]].sum(axis=1)
        return np.quantile(somas/len(diferenca),[.025,.975])

    def perda_fz0(perda,var,es,alpha):
        """Perda conjunta de quantil e cauda na convenção de perdas positivas."""
        perda,var,es = (np.asarray(x,dtype=float) for x in (perda,var,es))
        if (not 0<alpha<1 or perda.ndim!=1 or not len(perda) or
            perda.shape!=var.shape or perda.shape!=es.shape or not np.isfinite(np.c_[perda,var,es]).all()):
            raise ValueError("FZ0 exige vetores alinhados e finitos e nível de cauda entre zero e um.")
        if (var<=0).any() or (es<var).any():
            raise ValueError("FZ0 nesta implementação exige CVaR >= VaR > 0.")
        return np.maximum(perda-var,0)/(alpha*es)+var/es+np.log(es)-1

    def agrupamento_violacoes(violacoes,adjacentes,rep=5000,semente=42):
        """Teste unilateral de excesso de pares de violações contíguas."""
        violacoes = np.asarray(violacoes,dtype=bool)
        adjacentes = np.asarray(adjacentes,dtype=bool)
        if violacoes.ndim!=1 or not len(violacoes) or adjacentes.shape!=(len(violacoes)-1,) or rep<1:
            raise ValueError("Violações, adjacência ou número de repetições inválido.")
        esquerda = np.flatnonzero(adjacentes)
        total = int(violacoes.sum())
        pares = int((violacoes[esquerda]&violacoes[esquerda+1]).sum())
        if len(esquerda)==0 or total<2 or total==len(violacoes):
            return pares,np.nan
        rng = np.random.default_rng(semente)
        extremos = 0
        for _ in range(rep):
            simuladas = np.zeros(len(violacoes),dtype=bool)
            simuladas[rng.choice(len(violacoes),size=total,replace=False)] = True
            extremos += (simuladas[esquerda]&simuladas[esquerda+1]).sum()>=pares
        return pares,(extremos+1)/(rep+1)

validacao_cauda,comparacao_fz,linhas_violacoes = [],[],[]
for nivel in [95,99]:
    alpha = (100-nivel)/100
    cols = [f"{m}_{modelo}_{nivel}" for modelo in ["hist","t"] for m in ["VaR","ES"]]
    amostra = risco.loc[teste,cols].join(perda_d1.rename("perda")).dropna()
    adjacentes = adjacencia_sessoes(sessoes,amostra.index)
    perdas = amostra.perda.to_numpy()
    scores = {}
    for modelo in ["hist","t"]:
        v = amostra[f"VaR_{modelo}_{nivel}"].to_numpy()
        e = amostra[f"ES_{modelo}_{nivel}"].to_numpy()
        viola = perdas>v
        pares,p_agrupar = agrupamento_violacoes(viola,adjacentes,semente=SEED)
        total = int(viola.sum())
        cobertura = stats.binomtest(total,len(viola),alpha)
        ic = cobertura.proportion_ci()
        score = perda_fz0(perdas,v,e,alpha)
        scores[modelo] = score
        validacao_cauda.append({"nivel":nivel,"modelo":modelo,"n":len(viola),
            "violacoes":total,"violacoes_esperadas":alpha*len(viola),
            "taxa":viola.mean(),"taxa_ic_inf":ic.low,"taxa_ic_sup":ic.high,
            "p_cobertura_binomial":cobertura.pvalue,"pares_disponiveis":int(adjacentes.sum()),
            "pares_violacao":pares,"p_agrupamento":p_agrupar,"FZ0":score.mean()})
        linhas_violacoes.append(pd.DataFrame({"origem":amostra.index,"data_alvo":data_alvo.loc[amostra.index].to_numpy(),
            "nivel":nivel,"modelo":modelo,"violou":viola,"perda":perdas,"VaR":v,"CVaR":e,"FZ0":score}))
    diferenca = scores["t"]-scores["hist"]
    lo,hi = intervalo_media_pareada(diferenca,adjacentes,semente=SEED)
    comparacao_fz.append({"nivel":nivel,"n":len(diferenca),"FZ0_GARCH_menos_historico":diferenca.mean(),
                          "IC_inf":lo,"IC_sup":hi})
validacao_cauda = pd.DataFrame(validacao_cauda)
comparacao_fz = pd.DataFrame(comparacao_fz)
display(validacao_cauda)
display(comparacao_fz)
validacao_cauda.to_csv(SAIDA/"P1_backtest_cobertura_cauda.csv",index=False)
comparacao_fz.to_csv(SAIDA/"P1_comparacao_VaR_CVaR.csv",index=False)
pd.concat(linhas_violacoes,ignore_index=True).to_csv(SAIDA/"P1_violacoes_VaR_por_data.csv",index=False)
import shutil
zip_resultado = shutil.make_archive(str(SAIDA),"zip",SAIDA)
print("Arquivos para download:",zip_resultado)
